In [1]:
# !git clone https://github.com/goombalab/hnet.git   
# !git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
# # !git clone https://github.com/SDogya/congenial-adventure.git

# !rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure
# # !git clone https://github.com/SDogya/congenial-adventure.git
# # !cp  /kaggle/working/congenial-adventure/* /kaggle/working/

# import os
# import subprocess

# # Verify GPU
# result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
#                         capture_output=True, text=True)
# print('GPU:', result.stdout.strip())

# # W&B API key from Kaggle Secrets
# try:
#     from kaggle_secrets import UserSecretsClient
#     from kaggle_secrets import UserSecretsClient
#     user_secrets = UserSecretsClient()
#     secret_value_0 = user_secrets.get_secret("wandb")

#     secrets = UserSecretsClient()
#     os.environ['WANDB_API_KEY'] = user_secrets.get_secret("wandb")
#     print('W&B API key loaded from Kaggle Secrets')
# except Exception as e:
#     print(f'W&B secret not found ({e}), running in offline mode')
#     os.environ['WANDB_MODE'] = 'offline'

# WORKDIR = '/kaggle/working'
# os.chdir(WORKDIR)
# print('Working dir:', os.getcwd())

# !uv add wrapt zarr dill einops numba
# !uv sync 

# !rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure


# !uv run /kaggle/working/run.py

In [2]:
# !uv run /kaggle/working/run.py

In [3]:
# !uv add zarr dill einops numba vector-quantize-pytorch accelerate huggingface_hub wrapt

In [12]:
# ── 1. Repos & deps ──────────────────────────────────────────────────────────
!git clone https://github.com/goombalab/hnet.git
!git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
!rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure

import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('wandb')

!uv add "zarr<3.0.0" dill einops numba vector-quantize-pytorch accelerate huggingface_hub robomimic torchvision wrapt pillow pandas diffusers
!uv sync

# Patch OAT's lr_scheduler.py: newer diffusers removed Union/Optional/Optimizer exports
p = 'oat/oat/model/common/lr_scheduler.py'
txt = open(p).read()
marker = 'from diffusers.optimization import ('
if marker in txt and 'from typing import Union' not in txt:
    idx = txt.index(marker)
    end_idx = txt.index(')', idx) + 1
    header = (
        'from typing import Union, Optional\n'
        'from torch.optim import Optimizer\n'
        'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'
    )
    open(p, 'w').write(txt[:idx] + header + txt[end_idx:])
    print('lr_scheduler.py patched OK')
else:
    print('lr_scheduler.py already clean, skipping')




fatal: destination path 'hnet' already exists and is not an empty directory.
fatal: destination path 'oat' already exists and is not an empty directory.
Cloning into 'congenial-adventure'...
remote: Enumerating objects: 233, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 233 (delta 91), reused 189 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (233/233), 227.64 KiB | 10.35 MiB/s, done.
Resolving deltas: 100% (91/91), done.
Resolved 104 packages in 911ms                                       
Prepared 5 packages in 296ms                                             
Uninstalled 2 packages in 12ms
░░░░░░░░░░░░░░░░░░░░ [0/5] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is inte

In [5]:
# ── 2. Dataset (скачиваем прямо туда, куда смотрит OAT по дефолту) ───────────
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login
hf_token = UserSecretsClient().get_secret('hugface')

if hf_token:
    login(token=hf_token)
else:
    print("Ошибка: Секрет 'hugface' не найден.")


os.makedirs('/kaggle/working/oat/data/libero', exist_ok=True)
snapshot_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    repo_type='dataset',
    local_dir='/kaggle/working/oat/data/libero'
)



# Распаковываем архив прямо в ту же папку
!unzip -q /kaggle/working/oat/data/libero/libero10_N500.zarr.zip -d /kaggle/working/oat/data/libero/

# Проверяем, что папка появилась
!ls -ld /kaggle/working/oat/data/libero/*

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/kaggle/working/oat/data/libero'

In [13]:
# ── 3. Train OAT tokenizer (~2-3h, 300 epochs) ───────────────────────────────
# Запускаем из /kaggle/working — один venv, не создаём второй в oat/.venv
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment

Error executing job with overrides: ['task/tokenizer=libero/libero10', 'training.num_epochs=300', 'logging.project=VLA-experiment']
Error in call to target 'oat.dataset.zarr_dataset.ZarrDataset':
PathNotFoundError("nothing found at path ''")
full_key: task.tokenizer.dataset

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


In [23]:
# ── 3. Train OAT tokenizer (~2-3h, 300 epochs) ───────────────────────────────
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment \
    task.tokenizer.dataset.zarr_path="/kaggle/working/oat/data/libero/libero10_N500.zarr"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sebersehmer (sebersehmer-nopeinc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ setting up run l7nukmde (0.3s)
wandb: ⣷ setting up run l7nukmde (0.3s)
wandb: ⣯ setting up run l7nukmde (0.3s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/oat/output/20260412/175612_train_oattok_libero10_N500/wandb/run-20260412_175618-l7nukmde
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run 20260412.175612_train_oattok_libero10_N500
wandb: ⭐️ View project at https://wandb.ai/sebersehmer-nopeinc/VLA-experiment
wandb: 🚀 View run at https://wandb.ai/sebersehmer-nopeinc/VLA-experiment/runs/l7nukmde
Training epoch 111:   8%|▊        | 41/484 [00:06<01:11,  6.16it/s, los

In [41]:
# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
# Ищем последний чекпоинт (last.ckpt), исключая возможные чекпоинты самого FD-DRAT
!TOK=$(find /kaggle/working -path "*/*.ckpt" | tail -1) && \
 echo "Выбран чекпоинт токенизатора: $TOK" && \
 uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt=$TOK \
    dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
    batch_size=16

Выбран чекпоинт токенизатора: /kaggle/working/ep-0090_mse-0.004.ckpt
Could not override 'model.tokenizer_ckpt'.
To append to your config use +model.tokenizer_ckpt=/kaggle/working/ep-0090_mse-0.004.ckpt
Key 'tokenizer_ckpt' is not in struct
    full_key: model.tokenizer_ckpt
    object_type=dict

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


In [50]:
import yaml

# 1. Возвращаем визуальные ключи в конфигурацию
config_path = '/kaggle/working/conf/config.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['shape_meta'] = {
    'obs': {
        'agentview_rgb': {'type': 'rgb', 'shape': [128, 128, 3]},
        'robot0_eye_in_hand_rgb': {'type': 'rgb', 'shape': [128, 128, 3]},
        'robot0_eef_pos': {'type': 'state', 'shape': [3]},
        'robot0_eef_quat': {'type': 'state', 'shape': [4]},
        'robot0_gripper_qpos': {'type': 'state', 'shape': [2]},
        'task_uid': {'type': 'state', 'shape': [1]}
    },
    'action': {'shape': [7]}
}

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print("✅ Конфигурация восстановлена: визуальные ключи (RGB) добавлены.")

# 2. Патчим run.py (Monkey Patching для robomimic)
run_path = '/kaggle/working/run.py'
with open(run_path, 'r') as f:
    run_content = f.read()

patch = """
# --- ПАТЧ СОВМЕСТИМОСТИ API ROBOMIMIC ---
import robomimic.models.base_nets
import robomimic.models.obs_core
robomimic.models.base_nets.VisualCore = robomimic.models.obs_core.VisualCore
if hasattr(robomimic.models.obs_core, 'CropRandomizer'):
    robomimic.models.base_nets.CropRandomizer = robomimic.models.obs_core.CropRandomizer
# ----------------------------------------
import hydra"""

if "VisualCore" not in run_content:
    new_run_content = run_content.replace('import hydra', patch)
    with open(run_path, 'w') as f:
        f.write(new_run_content)
    print("✅ Скрипт run.py пропатчен: namespace robomimic.models.base_nets исправлен.")
else:
    print("⚡ Скрипт run.py уже пропатчен.")

✅ Конфигурация восстановлена: визуальные ключи (RGB) добавлены.
✅ Скрипт run.py пропатчен: namespace robomimic.models.base_nets исправлен.


In [47]:
# Доустанавливаем бэкенд для визуального энкодера
!uv add robomimic torchvision

# Если хочешь увидеть реальную ошибку импорта, выполни:
# !uv run python -c "from oat.perception.robomimic_vision_encoder import RobomimicRgbEncoder"

Resolved 124 packages in 1.66s                                       
Prepared 20 packages in 2.31s                                            
░░░░░░░░░░░░░░░░░░░░ [0/20] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 20 packages in 128ms                              
 + absl-py==2.4.0
 + contourpy==1.3.3
 + cycler==0.12.1
 + egl-probe==1.0.2
 + fonttools==4.62.1
 + grpcio==1.80.0
 + h5py==3.16.0
 + imageio==2.37.3
 + imageio-ffmpeg==0.6.0
 + kiwisolver==1.5.0
 + markdown==3.10.2
 + matplotlib==3.10.8
 + pyparsing==3.3.2
 + robomimic==0.3.0
 + tensorboard==2.20.0
 + tensorboard-data-server==0.7.2
 + tensorboardx==2.6.5
 + termcolor==3.3.0
 + torchvis

In [2]:
import yaml
import os

# 1. ПАТЧИМ СОВМЕСТИМОСТЬ API ROBOMIMIC (Monkey Patch)
run_path = '/kaggle/working/run.py'
if os.path.exists(run_path):
    with open(run_path, 'r') as f:
        run_content = f.read()
    
    patch = """
# --- ПАТЧ СОВМЕСТИМОСТИ API ROBOMIMIC ---
import robomimic.models.base_nets
import robomimic.models.obs_core
robomimic.models.base_nets.VisualCore = robomimic.models.obs_core.VisualCore
if hasattr(robomimic.models.obs_core, 'CropRandomizer'):
    robomimic.models.base_nets.CropRandomizer = robomimic.models.obs_core.CropRandomizer
# ----------------------------------------
import hydra"""
    
    if "VisualCore" not in run_content:
        new_run_content = run_content.replace('import hydra', patch)
        with open(run_path, 'w') as f:
            f.write(new_run_content)
        print("✅ Скрипт run.py пропатчен для robomimic.")

# 2. ЗАПУСК ОБУЧЕНИЯ С УМЕНЬШЕННЫМ БАТЧЕМ
# Обрати внимание: batch_size снижен до 4!
bash_cmd = """
TOK=$(find /kaggle/working -name "*.ckpt" | head -1) && \
if [ -z "$TOK" ]; then \
    echo "❌ ОШИБКА: Чекпоинт .ckpt не найден на диске! (Возможно, стерся при рестарте. Загрузи архив обратно.)"; \
else \
    echo "✅ Выбран чекпоинт: $TOK" && \
    env MPLBACKEND=Agg uv run run.py \
       strategy=single_gpu \
       +model.tokenizer_ckpt="$TOK" \
       +dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
       batch_size=4; \
fi
"""
os.system(bash_cmd)

✅ Выбран чекпоинт: /kaggle/working/ep-0100_mse-0.004.ckpt
ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /kaggle/working/.venv/lib/python3.12/site-packages/robomimic/scripts/setup_macros.py
)


Seed set to 42
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/kaggle/working/.venv/lib/python3.12/site-packages/lightning_fabric/connector.py:571: `precision=bf16` is supported for historical reasons but its usage is discouraged. Please set your precision to bf16-mixed instead!
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/projec

35072

In [3]:
import yaml
import os

print("🔧 Настраиваем параметры для экономии памяти Kaggle...")

# 1. ПАТЧИМ КОНФИГ (Выключаем параллельных воркеров)
config_path = '/kaggle/working/conf/config.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)
    
    # Жестко ограничиваем потребление памяти
    cfg['num_workers'] = 0 
    cfg['batch_size'] = 2  
    
    with open(config_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("✅ Установлено: num_workers=0, batch_size=2")

# 2. ПАТЧИМ ROBOMIMIC (Monkey Patch)
run_path = '/kaggle/working/run.py'
if os.path.exists(run_path):
    with open(run_path, 'r') as f:
        run_content = f.read()
    
    patch = """
# --- ПАТЧ СОВМЕСТИМОСТИ API ROBOMIMIC ---
import robomimic.models.base_nets
import robomimic.models.obs_core
robomimic.models.base_nets.VisualCore = robomimic.models.obs_core.VisualCore
if hasattr(robomimic.models.obs_core, 'CropRandomizer'):
    robomimic.models.base_nets.CropRandomizer = robomimic.models.obs_core.CropRandomizer
# ----------------------------------------
import hydra"""
    
    if "VisualCore" not in run_content:
        new_run_content = run_content.replace('import hydra', patch)
        with open(run_path, 'w') as f:
            f.write(new_run_content)
        print("✅ Скрипт run.py пропатчен для robomimic.")

# 3. ЗАПУСКАЕМ ОБУЧЕНИЕ
bash_cmd = """
TOK=$(find /kaggle/working -name "*.ckpt" | head -1) && \
if [ -z "$TOK" ]; then \
    echo "❌ ОШИБКА: Чекпоинт .ckpt не найден! Загрузи его на сервер."; \
else \
    echo "✅ Выбран чекпоинт: $TOK" && \
    env MPLBACKEND=Agg OMP_NUM_THREADS=1 uv run run.py \
       strategy=single_gpu \
       +model.tokenizer_ckpt="$TOK" \
       +dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr; \
fi
"""
os.system(bash_cmd)

🔧 Настраиваем параметры для экономии памяти Kaggle...
✅ Установлено: num_workers=0, batch_size=2
✅ Выбран чекпоинт: /kaggle/working/ep-0100_mse-0.004.ckpt
ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /kaggle/working/.venv/lib/python3.12/site-packages/robomimic/scripts/setup_macros.py
)


Seed set to 42
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/kaggle/working/.venv/lib/python3.12/site-packages/lightning_fabric/connector.py:571: `precision=bf16` is supported for historical reasons but its usage is discouraged. Please set your precision to bf16-mixed instead!
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/projec

35072

In [ ]:
import os

print("🔧 Полная замена datamodule.py (Устранение утечки памяти нормализатора)...")

datamodule_code = """
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader
from src.core.config_schema import ExperimentConfig
from oat.dataset.zarr_dataset import ZarrDataset

class LitDataModule(pl.LightningDataModule):
    def __init__(self, cfg: ExperimentConfig):
        super().__init__()
        self.cfg = cfg
        self.normalizer = None

    def setup(self, stage: str = None) -> None:
        all_obs_keys = list(self.cfg.shape_meta["obs"].keys())
        
        # --- ANTI-OOM HACK ---
        # Вытаскиваем только 1D-ключи, полностью игнорируя тяжелые картинки
        state_keys = [k for k, v in self.cfg.shape_meta["obs"].items() if v.get("type") != "rgb"]
        
        print(f"🛠️  Вычисляем нормализатор только для легких данных: {state_keys} + action")
        
        # 1. Временный датасет без картинок (потребляет < 100 МБ ОЗУ)
        temp_dataset = ZarrDataset(
            zarr_path=self.cfg.dataset_path,
            obs_keys=state_keys, 
            action_key='action',
            n_obs_steps=1,
            n_action_steps=self.cfg.model.H_a,
            val_ratio=0.1
        )
        self.normalizer = temp_dataset.get_normalizer()
        
        # Принудительно очищаем память от временного датасета
        del temp_dataset 
        
        print("✅ Нормализатор вычислен! Инициализируем основной датасет с ленивой загрузкой...")
        
        # 2. Боевой датасет со всеми ключами (картинки грузятся поштучно в getitem)
        self.train_dataset = ZarrDataset(
            zarr_path=self.cfg.dataset_path,
            obs_keys=all_obs_keys,
            action_key='action',
            n_obs_steps=1,
            n_action_steps=self.cfg.model.H_a,
            val_ratio=0.1
        )
        self.val_dataset = self.train_dataset.get_validation_dataset()

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset, 
            batch_size=self.cfg.batch_size,
            shuffle=True,
            num_workers=0,  # Строго 0 для Kaggle
            pin_memory=True
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset, 
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )
"""

# Перезаписываем файл
with open('/kaggle/working/src/core/datamodule.py', 'w') as f:
    f.write(datamodule_code.strip())
print("✅ Файл datamodule.py переписан.")

# Запуск
bash_cmd = """
TOK=$(find /kaggle/working -name "*.ckpt" | head -1) && \
if [ -z "$TOK" ]; then \
    echo "❌ ОШИБКА: Чекпоинт не найден!"; \
else \
    echo "✅ Выбран чекпоинт: $TOK" && \
    env MPLBACKEND=Agg OMP_NUM_THREADS=1 uv run run.py \
       strategy=single_gpu \
       +model.tokenizer_ckpt="$TOK" \
       +dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
       batch_size=4; \
fi
"""
os.system(bash_cmd)

🔧 Полная замена datamodule.py (Устранение утечки памяти нормализатора)...
✅ Файл datamodule.py переписан.
✅ Выбран чекпоинт: /kaggle/working/ep-0100_mse-0.004.ckpt
ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /kaggle/working/.venv/lib/python3.12/site-packages/robomimic/scripts/setup_macros.py
)


Seed set to 42
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/kaggle/working/.venv/lib/python3.12/site-packages/lightning_fabric/connector.py:571: `precision=bf16` is supported for historical reasons but its usage is discouraged. Please set your precision to bf16-mixed instead!
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/projec

ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /kaggle/working/.venv/lib/python3.12/site-packages/robomimic/scripts/setup_macros.py
)


[rank: 1] Seed set to 42
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [51]:
# ── 4. Train FD-DRAT (Full Vision Baseline) ──────────────────────────────────
!TOK=$(find /kaggle/working -name "*.ckpt" | head -1) && \
 if [ -z "$TOK" ]; then \
     echo "❌ ОШИБКА: Чекпоинт .ckpt не найден!"; \
 else \
     echo "✅ Выбран чекпоинт: $TOK" && \
     env MPLBACKEND=Agg uv run run.py \
        strategy=single_gpu \
        +model.tokenizer_ckpt="$TOK" \
        +dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
        batch_size=16; \
 fi

✅ Выбран чекпоинт: /kaggle/working/ep-0100_mse-0.004.ckpt
ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /kaggle/working/.venv/lib/python3.12/site-packages/robomimic/scripts/setup_macros.py
)
Seed set to 42
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/kaggle/working/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/kaggle/working/.venv/lib/python3.12/site-packages/lightning_fabric/connector.py:571: `precision=bf16` is supported for historical reasons but its usage is discouraged. Please set y

In [42]:
# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
!TOK=$(find /kaggle/working/oat/output -name 'latest.ckpt' | tail -1) && \
 uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt=$TOK \
    dataset_path=/kaggle/working/data/libero/libero10_N500.zarr \
    batch_size=16

Could not override 'model.tokenizer_ckpt'.
To append to your config use +model.tokenizer_ckpt=/kaggle/working/oat/output/20260412/175612_train_oattok_libero10_N500/checkpoints/latest.ckpt
Key 'tokenizer_ckpt' is not in struct
    full_key: model.tokenizer_ckpt
    object_type=dict

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


drwxrwxr-x 4 root root       4096 Jan 23 06:29 /kaggle/working/oat/data/libero/libero10_N500.zarr
-rw-r--r-- 1 root root 3528193175 Apr 12 17:32 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
drwxr-xr-x 3 root root       4096 Apr 12 17:53 /kaggle/working/oat/data/libero/__MACOSX
-rw-r--r-- 1 root root         24 Apr 12 17:32 /kaggle/working/oat/data/libero/README.md


In [38]:
import os
import zipfile
from IPython.display import FileLink, display

# 1. Рекурсивно ищем все файлы .ckpt
ckpt_files = []
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file.endswith('.ckpt'):
            ckpt_files.append(os.path.join(root, file))

if not ckpt_files:
    print("❌ ОШИБКА: Файлы .ckpt не найдены на диске! (возможно, они удалились при перезапуске)")
else:
    # 2. Берем самый последний найденный файл
    target_ckpt = ckpt_files[-1]
    print(f"✅ Найден чекпоинт: {target_ckpt}")
    
    zip_path = 'download_me.zip'
    print(f"📦 Упаковываем в {zip_path}...")
    
    # 3. Безопасно запаковываем его средствами Python
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(target_ckpt, arcname=os.path.basename(target_ckpt))
        
    print("🎯 Готово! Кликай по ссылке ниже:")
    display(FileLink(zip_path))

✅ Найден чекпоинт: /kaggle/working/oat/output/20260412/175612_train_oattok_libero10_N500/checkpoints/ep-0090_mse-0.004.ckpt
📦 Упаковываем в download_me.zip...
🎯 Готово! Кликай по ссылке ниже:


/kaggle/working/download_me.zip